In [1]:
import pandas as pd
import numpy as np
import os

from statsbombpy import sb
from statsbombpy import public

import duckdb
import random

#from action_types_config import ACTION_TYPES, TARGET_TYPE_DISTRIBUTION

In [9]:
PATH_DB = "../../00-data/statsbomb_event_data.duckdb"

## Read Rated Actions

In [7]:
#sheet_url = "https://docs.google.com/spreadsheets/d/1KNqlFw8iba9onasiJrtr8vRdsv3wCbus/edit?gid=655379371#gid=655379371"

#sheet_url = "https://docs.google.com/spreadsheets/d/1xXktsQ11NE-t3l-xTu70QeIPkXVTYSAp/edit?usp=sharing&ouid=117684318775327239607&rtpof=true&sd=true"

sheet_url = "https://docs.google.com/spreadsheets/d/1KUcLAxm5g-zz9myrTcQOLH5L09JmFCRaIJ0woCzY1TA/edit?usp=sharing"

sheet_id = sheet_url.split("/d/")[1].split("/")[0]

# Define sheet names or indices
#sheet_names = [0, 1]  # First two sheets

# Read data directly into pandas DataFrames
url_template = "https://docs.google.com/spreadsheets/d/{}/export?format=xlsx"

# Load the entire spreadsheet as an Excel file into memory
df_ratings = pd.read_excel(url_template.format(sheet_id), sheet_name=0)

## Load Event Details

In [10]:
ls_event_ids = df_ratings['id'].unique().tolist()

conn = duckdb.connect(PATH_DB)

ev_id_str = "', '".join(str(mid) for mid in ls_event_ids)
query = f"SELECT * FROM events WHERE id IN ('{ev_id_str}')"

df_actions_rated = conn.execute(query).fetchdf()
conn.close()

### Merge Event Details with Ratings

In [40]:
df_actions_rated.columns

Index(['id', 'match_id', 'period', 'minute', 'second', 'timestamp',
       'possession', 'possession_team', 'duration', 'team', 'player',
       'jersey_number', 'type', 'bodypart', 'start_x', 'start_y', 'end_x',
       'end_y', 'end_z', 'play_pattern', 'under_pressure', 'position',
       'shot_statsbomb_xg', 'shot_technique', 'shot_first_time',
       'shot_outcome', 'shot_type', 'pass_type', 'pass_angle', 'pass_length',
       'home_team', 'away_team', 'home_away', 'goal_home', 'goal_away',
       'home_score', 'away_score', 'scoreline', 'start_x_video',
       'start_y_video', 'end_x_video', 'end_y_video'],
      dtype='str')

In [41]:
df_ratings = pd.merge(df_ratings, df_actions_rated, on='id', how='left')

### Filter Actions not in selected Action Types

In [55]:
df_ratings[~df_ratings['type'].isin(['Pass', 'Dribble', 'Shot', 'Cross'])].action_not_recognized.value_counts()

action_not_recognized
False    739
True     259
Name: count, dtype: int64

### Identify Actions mostly not recognized

In [45]:
df_events = df_ratings.groupby('id').agg(
    num_ratings=('id', 'size'),
    all_not_recognized=('action_not_recognized', 'mean'),
    type =('type', 'first')
).reset_index()

In [56]:
df_events[(df_events.num_ratings > 2) &
          (df_events.all_not_recognized > .8)].type.value_counts()

type
Pass            26
Shot             9
Cross            4
throwin          3
Dribble          3
Clearance        2
Miscontrol       1
Foul Won         1
Dispossessed     1
Interception     1
Name: count, dtype: int64

In [36]:
df_actions_rated.type.value_counts()

type
Pass            2111
Dribble          867
Shot             315
Cross            290
Miscontrol       141
throwin          127
Clearance         85
Foul Won          54
Dispossessed      40
Interception      26
Error              3
Name: count, dtype: int64